In [13]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def get_soup(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

In [14]:
def detect_page_type(soup):
    if soup.find("table", class_="wikitable"):
        return "table"
    return "category"

In [15]:
def scrape_wikipedia_category_links(url):
    soup = get_soup(url)
    painting_links = []

    # Wikipedia category pages often store links in category groups
    content = soup.find("div", id="mw-pages")
    if not content:
        return painting_links

    for a in content.find_all("a", href=True):
        href = a["href"]
        full_url = urljoin("https://en.wikipedia.org", href)

        # skip navigation-ish links
        text = a.get_text(" ", strip=True)
        if not text or text.lower() in {"next page", "previous page"}:
            continue

        painting_links.append(full_url)

    # remove duplicates
    seen = set()
    unique_links = []
    for link in painting_links:
        if link not in seen:
            seen.add(link)
            unique_links.append(link)

    return unique_links

In [27]:
import re

def scrape_wikipedia_painting_page(painting_url):
    soup = get_soup(painting_url)

    title = None
    year = None
    image_url = None

    # title
    heading = soup.find("h1", id="firstHeading")
    if heading:
        title = heading.get_text(" ", strip=True)

    # image
    infobox = soup.find("table", class_="infobox")
    if infobox:
        img = infobox.find("img")
        if img and img.get("src"):
            image_url = urljoin("https:", img["src"])

        # try to find a year in infobox rows
        rows = infobox.find_all("tr")
        for row in rows:
            header = row.find("th")
            data = row.find("td")
            if not header or not data:
                continue

            htxt = header.get_text(" ", strip=True).lower()
            dtxt = data.get_text(" ", strip=True)

            if "year" in htxt or "date" in htxt:
                year = dtxt
                break

    # fallback: regex over page text
    if not year:
        text = soup.get_text(" ", strip=True)
        match = re.search(r"\b(1[5-9]\d{2}|20\d{2})\b", text)
        if match:
            year = match.group(1)

    return {
        "title": title,
        "year": year,
        "image_url": image_url,
        "painting_url": painting_url
    }

In [ ]:
from urllib.parse import urljoin

def scrape_wikipedia_paintings_table(url):
    soup = get_soup(url)
    paintings = []

    tables = soup.find_all("table", class_="wikitable")

    for table in tables:
        rows = table.find_all("tr")

        # skip header row
        for row in rows[1:]:
            cells = row.find_all("td")
            if len(cells) < 2:
                continue

            # ---- title + image usually live in first cell ----
            first_cell = cells[0]

            # image url
            image_url = None
            img_tag = first_cell.find("img")
            if img_tag and img_tag.get("src"):
                src = img_tag["src"]
                if src.startswith("//"):
                    image_url = "https:" + src
                else:
                    image_url = urljoin(url, src)

            # title
            title = None

            # try italicized title first
            italic = first_cell.find("i")
            if italic:
                title = italic.get_text(" ", strip=True)

            # fallback: first non-empty link text
            if not title:
                for a in first_cell.find_all("a", href=True):
                    txt = a.get_text(" ", strip=True)
                    if txt and txt.lower() not in {"edit", "view at"}:
                        title = txt
                        break

            # fallback: raw text from first cell
            if not title:
                title = first_cell.get_text(" ", strip=True)

            # ---- year usually lives in second cell ----
            year = cells[1].get_text(" ", strip=True) if len(cells) > 1 else None

            paintings.append({
                "title": title if title else None,
                "year": year if year else None,
                "image_url": image_url
            })

    return paintings

In [19]:
def scrape_wikipedia_artist_page(url, max_items=None):
    soup = get_soup(url)
    page_type = detect_page_type(soup)

    if page_type == "table":
        return scrape_wikipedia_paintings_table(url)

    # otherwise category/list page
    links = scrape_wikipedia_category_links(url)
    if max_items is not None:
        links = links[:max_items]

    paintings = []
    for link in links:
        try:
            paintings.append(scrape_wikipedia_painting_page(link))
            print(f"Scraped: {link}")
        except Exception as e:
            print(f"Failed: {link} -> {e}")

    return paintings

In [ ]:
def clean_paintings(paintings):
    cleaned = []

    for p in paintings:
        title = p.get("title")
        year_str = p.get("year")
        image_url = p.get("image_url")

        # skip None or empty values
        if not title or not year_str or not image_url:
            continue

        # parse year (first 4-digit number)
        match = re.search(r"\d{4}", str(year_str))
        if not match:
            continue

        year = int(match.group())

        cleaned.append({
            "title": title.strip(),
            "year": year,
            "image_url": image_url
        })

    return cleaned

In [35]:
# Test 1 with Andy Warhol, hyperlink structure, page does not directly provide the images

warhol_url = "https://en.wikipedia.org/wiki/Category:Paintings_by_Andy_Warhol"
paintings = scrape_wikipedia_artist_page(warhol_url, max_items=10)
clean_paintings_list = clean_paintings(paintings)

for p in clean_paintings_list:
    print(p)

Scraped: https://en.wikipedia.org/wiki/Wikipedia:FAQ/Categorization#Why_might_a_category_list_not_be_up_to_date?
Scraped: https://en.wikipedia.org/wiki/3_Coke_Bottles
Scraped: https://en.wikipedia.org/wiki/129_Die_in_Jet!
Scraped: https://en.wikipedia.org/wiki/Athletes_(Warhol_series)
Scraped: https://en.wikipedia.org/wiki/Big_Electric_Chair
Scraped: https://en.wikipedia.org/wiki/Camouflage_Self-Portrait
Scraped: https://en.wikipedia.org/wiki/Campbell%27s_Soup_Cans
Scraped: https://en.wikipedia.org/wiki/Campbell%27s_Soup_Cans_II
Scraped: https://en.wikipedia.org/wiki/Campbell%27s_Soup_I
Scraped: https://en.wikipedia.org/wiki/Cars_(painting)
{'title': 'Athletes (Warhol series)', 'year': '1977', 'image_url': 'https://upload.wikimedia.org/wikipedia/en/thumb/4/4c/Warhol-Athletes-1977.jpg/330px-Warhol-Athletes-1977.jpg', 'painting_url': 'https://en.wikipedia.org/wiki/Athletes_(Warhol_series)'}
{'title': 'Big Electric Chair', 'year': '1967', 'image_url': 'https://upload.wikimedia.org/wikiped

In [37]:
# Test 2 with Claude Monet, table structure, page directly provides the paintings

monet_url = "https://en.wikipedia.org/wiki/List_of_paintings_by_Claude_Monet#"
paintings = scrape_wikipedia_artist_page(monet_url, max_items=10)
clean_paintings_list = clean_paintings(paintings)

for p in clean_paintings_list:
    print(p)

{'title': 'View at Rouelles, Le Havre', 'year': '1858', 'image_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/f/fb/Monet_Veduta_di_Rouelles.jpg/250px-Monet_Veduta_di_Rouelles.jpg'}
{'title': 'Landscape with Factories', 'year': '1858–61', 'image_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/c/c1/Monet_-_Paysage_d%27usines_W5.jpg/250px-Monet_-_Paysage_d%27usines_W5.jpg'}
{'title': 'Landscape with Factories', 'year': '1858–61', 'image_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/d/d7/Monet_-_Wildenstein_1996%2C_5a.png/250px-Monet_-_Wildenstein_1996%2C_5a.png'}
{'title': 'Corner of a Studio', 'year': '1861', 'image_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/8/88/Coint_d%27atelier.jpg/250px-Coint_d%27atelier.jpg'}
{'title': 'Still Life with Pheasant', 'year': '1861', 'image_url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/f/fd/Monet-faisan-mus%C3%A9e_de_rouen.jpg/250px-Monet-faisan-mus%C3%A9e_de_rouen.jpg'}
{'title': 'La 